# Silver DQ — Great Expectations Validation
**CNG Distribution Analytics Platform**

Validates `Files/silverdata/` output before Gold is allowed to run.

## Checks
| Category | Check |
|---|---|
| Completeness | Row count > 0 |
| Completeness | Row count regression vs baseline |
| Schema | All expected columns present |
| Schema | No unexpected extra columns (WARN only) |
| Schema | Column count matches expected |
| Hygiene | Critical ID columns non-null |
| Hygiene | Audit columns present and fully populated |
| Silver-specific | Date columns parsed to datetime64 |
| Silver-specific | Categorical columns title-cased |
| Silver-specific | Numeric columns are float64 |
| Silver-specific | Boolean columns contain only True / False / None |

## 1. Config

In [3]:
import os, json
import pandas as pd
from datetime import datetime, timezone

SILVER_PATH   = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/"
BASELINE_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_baseline/"
DQ_LOG_PATH   = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_logs/"

for p in [BASELINE_PATH, DQ_LOG_PATH]:
    os.makedirs(p, exist_ok=True)

run_at          = datetime.now(timezone.utc)
pipeline_run_id = run_at.strftime('%Y%m%d_%H%M%S')

# ── Expected column counts (raw + Silver-derived + audit) ─────────────────────
# purchase_orders : 19 raw + 4 derived + 1 audit = 24
# sales_orders    : 17 raw + 3 derived + 1 audit = 21
# products        : 14 raw + 2 derived + 1 audit = 17
# customers       : 17 raw + 0 derived + 1 audit = 18
# inventory       : 17 raw + 0 derived + 1 audit = 18

TABLE_CONFIGS = [
    {
        "file"               : "silver_purchase_orders",
        "required_cols"      : ["POID", "ProductID", "SupplierID"],
        "date_cols"          : ["OrderDate", "ExpectedDeliveryDate", "ActualDeliveryDate"],
        "upper_cols"         : ["Status", "ApprovalStatus", "PortOfLoading"],
        "code_cols"          : ["IncoTerms", "Currency"],
        "numeric_cols"       : ["OrderedQty", "ReceivedQty", "UnitCost_USD",
                                "TotalCost_USD", "FreightCost_USD", "ExchangeRate"],
        "bool_cols"          : [],
        "expected_cols"      : ["POID", "SupplierID", "ProductID", "OrderedQty", "ReceivedQty",
                                "UnitCost_USD", "TotalCost_USD", "Currency", "ExchangeRate",
                                "OrderDate", "ExpectedDeliveryDate", "ActualDeliveryDate",
                                "Status", "IncoTerms", "PortOfLoading", "FreightCost_USD",
                                "BuyerName", "ApprovalStatus", "Notes",
                                "actual_lead_time_days", "expected_lead_time_days",
                                "is_late_delivery", "receipt_rate",
                                "_silver_transformed_at"],
        "expected_col_count" : 24,
    },
    {
        "file"               : "silver_sales_orders",
        "required_cols"      : ["OrderID", "CustomerID", "ProductID"],
        "date_cols"          : ["OrderDate", "RequestedDeliveryDate"],
        "upper_cols"         : ["Status", "Division", "SalesChannel"],
        "code_cols"          : [],
        "numeric_cols"       : ["OrderedQty", "UnitPrice_USD", "LineTotal_USD", "DiscountPct"],
        "bool_cols"          : ["IsRush"],
        "expected_cols"      : ["OrderID", "CustomerID", "ProductID", "Division",
                                "OrderedQty", "UnitPrice_USD", "LineTotal_USD", "DiscountPct",
                                "OrderDate", "RequestedDeliveryDate", "Status", "SalesChannel",
                                "SalesRepID", "POReference", "SpecialInstructions", "IsRush",
                                "SourceSystem",
                                "is_return", "discounted_unit_price", "calc_line_total_usd",
                                "_silver_transformed_at"],
        "expected_col_count" : 21,
    },
    {
        "file"               : "silver_products",
        "required_cols"      : ["ProductID"],
        "date_cols"          : ["IntroducedDate"],
        "upper_cols"         : ["Category", "SubCategory"],
        "code_cols"          : ["UOM"],
        "numeric_cols"       : ["BasePrice_USD", "StandardCost_USD", "HSCode", "WeightPerUnit_KG"],
        "bool_cols"          : ["IsHazardous", "ActiveFlag"],
        "expected_cols"      : ["ProductID", "ProductName", "Category", "SubCategory", "UOM",
                                "BasePrice_USD", "StandardCost_USD", "PrimarySupplierID",
                                "HSCode", "WeightPerUnit_KG", "IsHazardous", "Division",
                                "ActiveFlag", "IntroducedDate",
                                "gross_margin_pct", "is_below_cost",
                                "_silver_transformed_at"],
        "expected_col_count" : 17,
    },
    {
        "file"               : "silver_customers",
        "required_cols"      : ["CustomerID"],
        "date_cols"          : ["CustomerSince"],
        "upper_cols"         : ["Segment", "Region", "Division", "AnnualRevenueTier"],  # PaymentTerms removed — contains digits/slashes ("Net 30", "2/10 Net 30") that fail title-case regex
        "code_cols"          : ["PreferredCurrency"],
        "numeric_cols"       : ["CreditLimit_USD"],
        "bool_cols"          : ["ActiveFlag", "TaxExempt"],
        "expected_cols"      : ["CustomerID", "CustomerName", "Segment", "Division",
                                "Country", "State", "City", "Region", "CreditLimit_USD",
                                "PaymentTerms", "SalesRepID", "AccountManagerID",
                                "CustomerSince", "AnnualRevenueTier", "ActiveFlag",
                                "PreferredCurrency", "TaxExempt",
                                "_silver_transformed_at"],
        "expected_col_count" : 18,
    },
    {
        "file"               : "silver_inventory",
        "required_cols"      : ["InventoryID", "ProductID", "WarehouseID"],
        "date_cols"          : ["LastReceivedDate", "LastIssuedDate", "ExpiryDate", "LastUpdated"],
        "upper_cols"         : [],
        "code_cols"          : [],
        "numeric_cols"       : ["StockQty", "AllocatedQty", "AvailableQty",
                                "ReorderLevel", "ReorderQty", "UnitCost_USD", "TotalValue_USD"],
        "bool_cols"          : ["BelowReorder"],
        "expected_cols"      : ["InventoryID", "ProductID", "WarehouseID",
                                "StockQty", "AllocatedQty", "AvailableQty",
                                "ReorderLevel", "ReorderQty", "UnitCost_USD", "TotalValue_USD",
                                "LastReceivedDate", "LastIssuedDate", "LotNumber",
                                "ExpiryDate", "BelowReorder", "StorageLocation", "LastUpdated",
                                "_silver_transformed_at"],
        "expected_col_count" : 18,
    },
]

print(f"Config loaded   : {len(TABLE_CONFIGS)} tables")
print(f"Silver path     : {SILVER_PATH}")
print(f"Baseline path   : {BASELINE_PATH}")
print(f"DQ log path     : {DQ_LOG_PATH}")
print(f"Pipeline run ID : {pipeline_run_id}")

Config loaded   : 5 tables
Silver path     : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/silverdata/
Baseline path   : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_baseline/
DQ log path     : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_logs/
Pipeline run ID : 20260625_213143


## 2. Imports

In [4]:
# Great Expectations is pre-installed via the Fabric Environment (Public Libraries → PyPI).
# If running locally: pip install great-expectations
!pip install great-expectations
import great_expectations as gx
import great_expectations.expectations as gxe

print(f"great_expectations : {gx.__version__}")
print(f"pandas             : {pd.__version__}")

context = gx.get_context(mode="ephemeral")
print("GX context ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [great-expectations]reat-expectations]
great_expectations : 1.18.1
pandas             : 2.3.3
GX context ready.


## 3. Baseline helpers
Baselines persist the row count from the **first clean run** to disk (`dq_baseline/`).
Subsequent runs compare against it and FAIL if rows drop by more than 10%.

In [5]:
def load_baseline(table_name: str) -> dict | None:
    """Return the stored baseline dict, or None if it doesn't exist yet."""
    path = BASELINE_PATH + f"{table_name}_baseline.json"
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None


def save_baseline(table_name: str, row_count: int) -> None:
    """Persist row count as the baseline for future regression checks."""
    path = BASELINE_PATH + f"{table_name}_baseline.json"
    payload = {
        "table"    : table_name,
        "row_count": row_count,
        "saved_at" : run_at.isoformat(),
    }
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"  Baseline saved  : {row_count:,} rows -> {path}")

## 4. `build_suite` — GX expectation suite

Builds the full expectation suite for one Silver table.

| Expectation | Category |
|---|---|
| `ExpectTableRowCountToBeGreaterThan(0)` | Completeness |
| `ExpectTableRowCountToBeGreaterThan(baseline * 0.90)` | Completeness — regression |
| `ExpectTableColumnsToMatchSet(exact_match=False)` | Schema — expected cols present |
| `ExpectTableColumnCountToEqual(n)` | Schema — count matches |
| `ExpectColumnToExist` + `ExpectColumnValuesToNotBeNull` per required_col | Hygiene |
| `ExpectColumnToExist` + `ExpectColumnValuesToNotBeNull` for `_silver_transformed_at` | Hygiene — audit |
| `ExpectColumnValuesToBeOfType("datetime64[ns]")` per date_col | Silver-specific |
| `ExpectColumnValuesToMatchRegex` title-case per upper_col | Silver-specific |
| `ExpectColumnValuesToMatchRegex` ALL-CAPS per code_col | Silver-specific |
| `ExpectColumnValuesToBeOfType("float64")` per numeric_col | Silver-specific |
| `ExpectColumnValuesToBeInSet([True, False, None])` per bool_col | Silver-specific |


In [6]:
def build_suite(suite_name: str, df: pd.DataFrame, cfg: dict, baseline: dict | None):
    """
    Builds the GX expectation suite for one Silver table.
    Returns (suite, validation_definition).

    Idempotent: deletes any prior GX objects with the same name first so the
    cell can be re-run in the same notebook session without conflicts.

    GX version note: uses GX 1.x API.
    ExpectTableRowCountToBeGreaterThan was removed in GX 1.x;
    replaced by ExpectTableRowCountToBeBetween(min_value=n, strict_min=True).
    """
    # Clean up prior objects from earlier runs in this kernel session
    for collection, name in [
        (context.validation_definitions, f"vd_{suite_name}"),
        (context.suites,                 suite_name),
        (context.data_sources,           f"ds_{suite_name}"),
    ]:
        try:
            collection.delete(name=name)
        except Exception:
            pass

    # GX plumbing
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))
    ds    = context.data_sources.add_pandas(name=f"ds_{suite_name}")
    da    = ds.add_dataframe_asset(name=f"asset_{suite_name}")
    bd    = da.add_batch_definition_whole_dataframe(name=f"batch_{suite_name}")
    vd    = context.validation_definitions.add(
                gx.ValidationDefinition(name=f"vd_{suite_name}", data=bd, suite=suite)
            )

    # 1. COMPLETENESS - row count > 0
    # GX 1.x: use ExpectTableRowCountToBeBetween with strict_min=True instead of
    # the removed ExpectTableRowCountToBeGreaterThan
    suite.add_expectation(
        gxe.ExpectTableRowCountToBeBetween(min_value=0, strict_min=True)
    )

    # 2. COMPLETENESS - row count regression vs baseline (90% floor)
    if baseline:
        min_rows = int(baseline["row_count"] * 0.90)
        suite.add_expectation(
            gxe.ExpectTableRowCountToBeBetween(min_value=min_rows, strict_min=False)
        )

    # 3. SCHEMA - all expected columns present (extra columns: WARN only)
    suite.add_expectation(
        gxe.ExpectTableColumnsToMatchSet(
            column_set=cfg["expected_cols"],
            exact_match=False,
        )
    )

    # 4. SCHEMA - column count matches expected
    suite.add_expectation(
        gxe.ExpectTableColumnCountToEqual(value=cfg["expected_col_count"])
    )

    # 5. HYGIENE - required ID columns exist and are non-null
    for col in cfg["required_cols"]:
        suite.add_expectation(gxe.ExpectColumnToExist(column=col))
        suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column=col))

    # 6. HYGIENE - Silver audit column present and fully populated
    suite.add_expectation(gxe.ExpectColumnToExist(column="_silver_transformed_at"))
    suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="_silver_transformed_at"))

    # 7. SILVER-SPECIFIC - date columns are datetime64[ns]
    for col in cfg["date_cols"]:
        if col in df.columns:
            suite.add_expectation(
                gxe.ExpectColumnValuesToBeOfType(column=col, type_="datetime64[ns]")
            )

    # 8. SILVER-SPECIFIC - categorical cols are title-cased
    # Matches: "Approved", "Fully Received", "North American Distribution"
    # mostly=0.95 gives 5% tolerance for nulls-as-string edge cases
    # Updated: allow hyphens and slashes within words for "E-Commerce", "Office/Corporate"
    TITLE_REGEX = "^[A-Z][a-zA-Z/\\-]*( [A-Z][a-zA-Z/\\-]*)*$"
    for col in cfg["upper_cols"]:
        if col in df.columns:
            suite.add_expectation(
                gxe.ExpectColumnValuesToMatchRegex(
                    column=col,
                    regex=TITLE_REGEX,
                    mostly=0.95,
                )
            )

    # 9. SILVER-SPECIFIC - code cols are ALL-CAPS (e.g. "USD", "FOB", "MT")
    CODE_REGEX = "^[A-Z0-9/ -]+$"
    for col in cfg["code_cols"]:
        if col in df.columns:
            suite.add_expectation(
                gxe.ExpectColumnValuesToMatchRegex(
                    column=col,
                    regex=CODE_REGEX,
                    mostly=0.95,
                )
            )

    # 10. SILVER-SPECIFIC - numeric cols are float64
    for col in cfg["numeric_cols"]:
        if col in df.columns:
            suite.add_expectation(
                gxe.ExpectColumnValuesToBeOfType(column=col, type_="float64")
            )

    # 11. SILVER-SPECIFIC - boolean cols contain only True / False / None
    for col in cfg["bool_cols"]:
        if col in df.columns:
            suite.add_expectation(
                gxe.ExpectColumnValuesToBeInSet(
                    column=col,
                    value_set=[True, False, None],
                    mostly=1.0,
                )
            )

    return suite, vd

## 5. Run validation loop

In [7]:
import numpy as np

dq_results = []   # one row per table — written to the DQ log at the end
hard_fails = []   # tables that FAIL and must block the Gold run

for cfg in TABLE_CONFIGS:
    table = cfg["file"]
    print(f"\n{'='*60}")
    print(f"Validating: {table}")
    print(f"{'='*60}")

    # ── Load Silver CSV ───────────────────────────────────────────────────────
    try:
        silver_file = SILVER_PATH + table + ".csv"
        df = pd.read_csv(silver_file)

        # Silver writes pd.NA as the literal string "<Na>" when saving to CSV.
        # Replace with NaN so GX treats it as missing (skipped by mostly=)
        # rather than a non-null string that fails regex checks.
        df = df.replace("<Na>", np.nan)

        # Re-parse date columns so GX sees datetime64[ns], not object
        for col in cfg["date_cols"]:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        # Re-cast numeric columns to float64 (CSV round-trip may widen to object)
        for col in cfg["numeric_cols"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

        print(f"  Loaded : {len(df):,} rows, {len(df.columns)} cols")

    except Exception as e:
        print(f"  ERROR loading file: {e}")
        dq_results.append({
            "table"           : table,
            "status"          : "LOAD_ERROR",
            "row_count"       : 0,
            "baseline_rows"   : None,
            "checks_passed"   : 0,
            "checks_failed"   : 1,
            "failure_details" : str(e),
            "run_at"          : run_at.isoformat(),
            "pipeline_run_id" : pipeline_run_id,
        })
        hard_fails.append(table)
        continue

    # ── Load (or seed) baseline ───────────────────────────────────────────────
    baseline = load_baseline(table)
    if baseline is None:
        print(f"  No baseline found — seeding from this run ({len(df):,} rows).")
        save_baseline(table, len(df))
        baseline = None   # skip regression check on the seed run

    # ── Build suite + run validation ──────────────────────────────────────────
    suite, vd = build_suite(
        suite_name=f"suite_{table}",
        df=df,
        cfg=cfg,
        baseline=baseline,
    )

    result = vd.run(batch_parameters={"dataframe": df})

    # ── Parse results ─────────────────────────────────────────────────────────
    passed = sum(1 for r in result.results if r.success)
    failed = sum(1 for r in result.results if not r.success)
    total  = len(result.results)

    print(f"  Checks : {total} total  |  {passed} passed  |  {failed} failed")

    failure_msgs = []
    for r in result.results:
        exp_type = r.expectation_config.type
        if not r.success:
            detail = r.result.get("observed_value", r.result)
            msg = f"FAIL [{exp_type}] observed: {detail}"
            print(f"  {msg}")
            failure_msgs.append(msg)
        else:
            detail = r.result.get("observed_value", "")
            print(f"  [OK ] [{exp_type}] {detail}")

    # ── Determine overall status ──────────────────────────────────────────────
    status = "PASS" if failed == 0 else "FAIL"
    if status == "FAIL":
        hard_fails.append(table)

    # ── Update baseline only when this run passed ─────────────────────────────
    if status == "PASS":
        save_baseline(table, len(df))

    dq_results.append({
        "table"           : table,
        "status"          : status,
        "row_count"       : len(df),
        "baseline_rows"   : baseline["row_count"] if baseline else len(df),
        "checks_passed"   : passed,
        "checks_failed"   : failed,
        "failure_details" : " | ".join(failure_msgs) if failure_msgs else None,
        "run_at"          : run_at.isoformat(),
        "pipeline_run_id" : pipeline_run_id,
    })

print(f"\n{'='*60}")
print("SILVER DQ COMPLETE")
print(f"{'='*60}")


Validating: silver_purchase_orders
  Loaded : 1,144 rows, 24 cols
  No baseline found — seeding from this run (1,144 rows).
  Baseline saved  : 1,144 rows -> abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_baseline/silver_purchase_orders_baseline.json


Calculating Metrics:   0%|          | 0/61 [00:00<?, ?it/s]

/home/trusted-service-user/jupyter-env/python3.12/lib/python3.12/site-packages/great_expectations/expectations/metrics/column_map_metrics/column_values_match_regex.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return column.astype(str).str.contains(regex)

/home/trusted-service-user/jupyter-env/python3.12/lib/python3.12/site-packages/great_expectations/expectations/metrics/column_map_metrics/column_values_match_regex.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return column.astype(str).str.contains(regex)

/home/trusted-service-user/jupyter-env/python3.12/lib/python3.12/site-packages/great_expectations/expectations/metrics/column_map_metrics/column_values_match_regex.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str

  Checks : 25 total  |  25 passed  |  0 failed
  [OK ] [expect_table_row_count_to_be_between] 1144
  [OK ] [expect_table_columns_to_match_set] ['ActualDeliveryDate', 'ApprovalStatus', 'BuyerName', 'Currency', 'ExchangeRate', 'ExpectedDeliveryDate', 'FreightCost_USD', 'IncoTerms', 'Notes', 'OrderDate', 'OrderedQty', 'POID', 'PortOfLoading', 'ProductID', 'ReceivedQty', 'Status', 'SupplierID', 'TotalCost_USD', 'UnitCost_USD', '_silver_transformed_at', 'actual_lead_time_days', 'expected_lead_time_days', 'is_late_delivery', 'receipt_rate']
  [OK ] [expect_table_column_count_to_equal] 24
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_values_t

Calculating Metrics:   0%|          | 0/55 [00:00<?, ?it/s]

...

  Checks : 17 total  |  17 passed  |  0 failed
  [OK ] [expect_table_row_count_to_be_between] 40
  [OK ] [expect_table_columns_to_match_set] ['ActiveFlag', 'BasePrice_USD', 'Category', 'Division', 'HSCode', 'IntroducedDate', 'IsHazardous', 'PrimarySupplierID', 'ProductID', 'ProductName', 'StandardCost_USD', 'SubCategory', 'UOM', 'WeightPerUnit_KG', '_silver_transformed_at', 'gross_margin_pct', 'is_below_cost']
  [OK ] [expect_table_column_count_to_equal] 17
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_be_of_type] float64
  [OK ] [expect_column_values_to_be_of_type] float64
  [OK ] [expect_column_values_to_be_of_type] float64
  [OK ] [expect_colu

Calculating Metrics:   0%|          | 0/66 [00:00<?, ?it/s]

  Checks : 16 total  |  16 passed  |  0 failed
  [OK ] [expect_table_row_count_to_be_between] 70
  [OK ] [expect_table_columns_to_match_set] ['AccountManagerID', 'ActiveFlag', 'AnnualRevenueTier', 'City', 'Country', 'CreditLimit_USD', 'CustomerID', 'CustomerName', 'CustomerSince', 'Division', 'PaymentTerms', 'PreferredCurrency', 'Region', 'SalesRepID', 'Segment', 'State', 'TaxExempt', '_silver_transformed_at']
  [OK ] [expect_table_column_count_to_equal] 18
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_match_regex] 
  [OK ] [expect_column_values_to_be_of_type] float64
  [OK ] [expect_column_values_to

Calculating Metrics:   0%|          | 0/34 [00:00<?, ?it/s]

  Checks : 23 total  |  23 passed  |  0 failed
  [OK ] [expect_table_row_count_to_be_between] 187
  [OK ] [expect_table_columns_to_match_set] ['AllocatedQty', 'AvailableQty', 'BelowReorder', 'ExpiryDate', 'InventoryID', 'LastIssuedDate', 'LastReceivedDate', 'LastUpdated', 'LotNumber', 'ProductID', 'ReorderLevel', 'ReorderQty', 'StockQty', 'StorageLocation', 'TotalValue_USD', 'UnitCost_USD', 'WarehouseID', '_silver_transformed_at']
  [OK ] [expect_table_column_count_to_equal] 18
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_to_exist] 
  [OK ] [expect_column_values_to_not_be_null] 
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_values_to_be_of_type] datetime64
  [OK ] [expect_column_va

## 6. Write DQ log & halt pipeline on failure

In [8]:
# ── Print summary table ───────────────────────────────────────────────────────
summary_df = pd.DataFrame(dq_results)
print(summary_df[["table", "status", "row_count", "baseline_rows",
                   "checks_passed", "checks_failed"]].to_string(index=False))

# ── Persist DQ log ────────────────────────────────────────────────────────────
log_path = DQ_LOG_PATH + f"silver_dq_{pipeline_run_id}.csv"
summary_df.to_csv(log_path, index=False)
print(f"\nDQ log written -> {log_path}")

# ── Halt pipeline on failure ──────────────────────────────────────────────────
# The Fabric Data Factory pipeline has an 'on success' dependency between this
# notebook activity and the Gold notebook activity.
# Raising an exception marks this activity as Failed, blocking Gold from running.
# Same pattern used in Bronze ingestion notebook.
if hard_fails:
    raise Exception(
        f"Silver DQ FAILED for {len(hard_fails)} table(s): {hard_fails}. "
        "Gold notebook will not run."
    )
else:
    print("\nAll Silver tables passed DQ. Safe to run Gold notebook.")

                 table status  row_count  baseline_rows  checks_passed  checks_failed
silver_purchase_orders   PASS       1144           1144             25              0
   silver_sales_orders   PASS       4864           4864             21              0
       silver_products   PASS         40             40             17              0
      silver_customers   PASS         70             70             16              0
      silver_inventory   PASS        187            187             23              0

DQ log written -> abfss://CNG_project@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Files/dq_logs/silver_dq_20260625_213143.csv

All Silver tables passed DQ. Safe to run Gold notebook.
